# Introduction to Linear Model

### Student Performance (Multiple Linear Regression)

Exploring Factors Affecting Student Performance

https://www.kaggle.com/datasets/nikhil7280/student-performance-multiple-linear-regression

In [1]:
from utils import *

# Suppress warnings
warnings.filterwarnings("ignore")

datasets = {"Studant Performance": ["student-perf/Student_Performance.csv", "Performance Index"],
            "Flood Prediction": ["playground-series-s4e5/train.csv", "FloodProbability"],
            "Abalone Rings": ["playground-series-s4e4/train.csv", "Rings"],}

result = {}
for dataset_name, (file_path, target_column) in datasets.items():
    # Load the dataset
    df = pd.read_csv(f"../data/{file_path}")
    result[dataset_name] = run_models(df, dataset=dataset_name ,target=target_column)

pd.DataFrame(result).to_csv("./results/benchmark_results.csv", index=False)
print("Benchmarking completed. Results saved to '../results/benchmark_results.csv'.")

LinearRegressionOLS MSE: 4.0826, Train Size: 8000, Training time: 0.00 seconds
LinearRegressionSGD MSE: 4.1450, Train Size: 8000, Training time: 0.25 seconds
LinearRegressionRidge MSE: 4.0826, Train Size: 8000, Training time: 0.00 seconds
LinearRegressionLasso MSE: 4.1422, Train Size: 8000, Training time: 0.00 seconds
LinearRegressionElasticNet MSE: 4.1178, Train Size: 8000, Training time: 0.00 seconds
KNN MSE: 5.9776, Train Size: 8000, Training time: 0.01 seconds
DecisionTree MSE: 8.8291, Train Size: 8000, Training time: 0.02 seconds
SVM MSE: 205.6060, Train Size: 8000, Training time: 0.51 seconds
RandomForest MSE: 5.1588, Train Size: 8000, Training time: 0.97 seconds
XGBoost MSE: 4.9258, Train Size: 8000, Training time: 0.19 seconds
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
LightGBM MSE: 4.3094, Train Size: 8000, Training time: 0.10 seconds
CatBoost MSE: 4.3294, Train Size: 8000, Training time: 0.22 seconds
LinearRegressionOLS MSE: 0.0004, Train 

In [4]:
pd.DataFrame(result)

,Studant Performance,Flood Prediction,Abalone Rings
LinearRegressionOLS,"{'Test MSE': 4.082628398521853, 'Size': 8000, ...","{'Test MSE': 0.00040691328774306767, 'Size': 4...","{'Test MSE': 4.162742327018618, 'Size': 72492,..."
LinearRegressionSGD,"{'Test MSE': 4.144994558818909, 'Size': 8000, ...","{'Test MSE': 2.4587103341172698e+33, 'Size': 4...","{'Test MSE': 2.529726252488269e+27, 'Size': 72..."
LinearRegressionRidge,"{'Test MSE': 4.082629374295429, 'Size': 8000, ...","{'Test MSE': 0.00040691328673496445, 'Size': 4...","{'Test MSE': 4.16308139116776, 'Size': 72492, ..."
LinearRegressionLasso,"{'Test MSE': 4.1422437278361635, 'Size': 8000,...","{'Test MSE': 0.0026017298235197278, 'Size': 40...","{'Test MSE': 6.447188859990774, 'Size': 72492,..."
LinearRegressionElasticNet,"{'Test MSE': 4.1178310328089225, 'Size': 8000,...","{'Test MSE': 0.002601729952271495, 'Size': 400...","{'Test MSE': 6.502457081584579, 'Size': 72492,..."
KNN,"{'Test MSE': 5.977620000000001, 'Size': 8000, ...","{'Test MSE': 0.0028451829399999996, 'Size': 40...","{'Test MSE': 12.04389118799316, 'Size': 72492,..."
DecisionTree,"{'Test MSE': 8.829069444444444, 'Size': 8000, ...","{'Test MSE': 0.0025176661660115253, 'Size': 40...","{'Test MSE': 6.585969799841058, 'Size': 72492,..."
SVM,"{'Test MSE': 205.60596160380908, 'Size': 8000,...","{'Test MSE': 0.003985929362825388, 'Size': 400...","{'Test MSE': 10.364924146118291, 'Size': 72492..."
RandomForest,"{'Test MSE': 5.158838006796808, 'Size': 8000, ...","{'Test MSE': 0.0009518266248500002, 'Size': 40...","{'Test MSE': 3.621665347900459, 'Size': 72492,..."
XGBoost,"{'Test MSE': 4.925849899013449, 'Size': 8000, ...","{'Test MSE': 0.000509474726485801, 'Size': 400...","{'Test MSE': 3.551525115966797, 'Size': 72492,..."


In [33]:
report = ProfileReport(
    df,
    title="Student Performance Report",
    explorative=True,
    progress_bar=False)
report.to_file("../reports/flood_prediction.html")

100%|██████████| 22/22 [00:01<00:00, 16.09it/s]


#### Neural Networks

In [39]:
import torch
from sklearn.preprocessing import StandardScaler

import torch.nn as nn
import torch.optim as optim

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# Define the MLP model
class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x

# Initialize the model, loss function, and optimizer
input_size = X_train.shape[1]
model = MLP(input_size)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train the model
epochs = 1000
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    y_pred = model(X_train_tensor)
    loss = criterion(y_pred, y_train_tensor)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.4f}")

# Evaluate the model
model.eval()
with torch.no_grad():
    y_test_pred = model(X_test_tensor)
    test_loss = criterion(y_test_pred, y_test_tensor)
    print(f"Test MSE: {test_loss.item():.4f}")

Epoch 100/1000, Loss: 1930.5172
Epoch 200/1000, Loss: 90.5274
Epoch 300/1000, Loss: 52.4664
Epoch 400/1000, Loss: 42.8137
Epoch 500/1000, Loss: 35.5108
Epoch 600/1000, Loss: 29.0945
Epoch 700/1000, Loss: 23.5786
Epoch 800/1000, Loss: 19.0498
Epoch 900/1000, Loss: 15.4358
Epoch 1000/1000, Loss: 12.5401
Test MSE: 12.5687


In [40]:
import torch
from sklearn.preprocessing import StandardScaler

import torch.nn as nn
import torch.optim as optim

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# Reshape input for LSTM (batch_size, sequence_length, input_size)
X_train_tensor = X_train_tensor.unsqueeze(1)
X_test_tensor = X_test_tensor.unsqueeze(1)

# Define the LSTM model
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])  # Take the output of the last time step
        return out

# Initialize the model, loss function, and optimizer
input_size = X_train.shape[1]
hidden_size = 64
num_layers = 2
model = LSTMModel(input_size, hidden_size, num_layers)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train the model
epochs = 1000
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    y_pred = model(X_train_tensor)
    loss = criterion(y_pred, y_train_tensor)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.4f}")

# Evaluate the model
model.eval()
with torch.no_grad():
    y_test_pred = model(X_test_tensor)
    test_loss = criterion(y_test_pred, y_test_tensor)
    print(f"Test MSE: {test_loss.item():.4f}")

Epoch 100/1000, Loss: 2905.7173
Epoch 200/1000, Loss: 1926.2614
Epoch 300/1000, Loss: 1470.3459
Epoch 400/1000, Loss: 1163.2555
Epoch 500/1000, Loss: 937.1789
Epoch 600/1000, Loss: 759.9985
Epoch 700/1000, Loss: 613.5988
Epoch 800/1000, Loss: 501.2657
Epoch 900/1000, Loss: 412.1020
Epoch 1000/1000, Loss: 339.8988
Test MSE: 329.1992


In [41]:
import torch
from sklearn.preprocessing import StandardScaler

import torch.nn as nn
import torch.optim as optim

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# Define the Transformer model
class TransformerModel(nn.Module):
    def __init__(self, input_size, d_model, nhead, num_layers, dim_feedforward, dropout=0.1):
        super(TransformerModel, self).__init__()
        self.input_layer = nn.Linear(input_size, d_model)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.output_layer = nn.Linear(d_model, 1)

    def forward(self, src):
        src = self.input_layer(src)
        src = self.transformer(src, src)
        output = self.output_layer(src[:, -1, :])  # Use the last time step
        return output

# Initialize the model, loss function, and optimizer
input_size = X_train.shape[1]
d_model = 64
nhead = 4
num_layers = 2
dim_feedforward = 128
model = TransformerModel(input_size, d_model, nhead, num_layers, dim_feedforward)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train the model
epochs = 1000
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    y_pred = model(X_train_tensor.unsqueeze(1))  # Add sequence dimension
    loss = criterion(y_pred, y_train_tensor)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.4f}")

# Evaluate the model
model.eval()
with torch.no_grad():
    y_test_pred = model(X_test_tensor.unsqueeze(1))  # Add sequence dimension
    test_loss = criterion(y_test_pred, y_test_tensor)
    print(f"Test MSE: {test_loss.item():.4f}")

Epoch 100/1000, Loss: 2184.0544
Epoch 200/1000, Loss: 1278.3195
Epoch 300/1000, Loss: 653.5947
Epoch 400/1000, Loss: 282.1443
Epoch 500/1000, Loss: 126.9255
Epoch 600/1000, Loss: 62.7932
Epoch 700/1000, Loss: 35.0438
Epoch 800/1000, Loss: 21.9157
Epoch 900/1000, Loss: 15.2010
Epoch 1000/1000, Loss: 11.5821
Test MSE: 10.3565


In [7]:
import optuna
from sklearn.model_selection import train_test_split
from xgboost import DMatrix, train as xgb_train

# Split the training data into a smaller training set and a validation set for early stopping
X_train_sub, X_val, y_train_sub, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Define the objective function for Optuna
def objective(trial):
    # Suggest hyperparameters
    max_depth = trial.suggest_int("max_depth", 3, 10)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3, log=True)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    reg_alpha = trial.suggest_float("reg_alpha", 0.0, 10.0)
    reg_lambda = trial.suggest_float("reg_lambda", 0.0, 10.0)

    # Prepare the data for XGBoost
    dtrain = DMatrix(X_train_sub, label=y_train_sub)
    dval = DMatrix(X_val, label=y_val)
    dtest = DMatrix(X_test)

    # Define the parameters
    params = {
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "subsample": subsample,
        "colsample_bytree": colsample_bytree,
        "reg_alpha": reg_alpha,
        "reg_lambda": reg_lambda,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "seed": 42
    }

    # Train the model with early stopping
    evals = [(dtrain, "train"), (dval, "eval")]
    model = xgb_train(params, dtrain, num_boost_round=10000, evals=evals, early_stopping_rounds=50, verbose_eval=False)

    # Predict on the test set
    y_pred = model.predict(dtest)

    # Calculate the mean squared error
    mse = mean_squared_error(y_test, y_pred)
    return mse

# Create a study and optimize
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.HyperbandPruner())
study.optimize(objective, n_trials=50)

# Print the best hyperparameters and the corresponding MSE
print("Best hyperparameters:", study.best_params)
print("Best MSE:", study.best_value)

[I 2025-06-22 14:23:18,661] A new study created in memory with name: no-name-e372fe4f-2381-4112-9621-6736f60a7270
[I 2025-06-22 14:23:19,378] Trial 0 finished with value: 4.501244594802438 and parameters: {'max_depth': 5, 'learning_rate': 0.2536999076681772, 'subsample': 0.8659969709057025, 'colsample_bytree': 0.7993292420985183, 'reg_alpha': 1.5601864044243652, 'reg_lambda': 1.5599452033620265}. Best is trial 0 with value: 4.501244594802438.
[I 2025-06-22 14:23:19,743] Trial 1 finished with value: 4.3050510008551885 and parameters: {'max_depth': 3, 'learning_rate': 0.19030368381735815, 'subsample': 0.8005575058716043, 'colsample_bytree': 0.8540362888980227, 'reg_alpha': 0.20584494295802447, 'reg_lambda': 9.699098521619943}. Best is trial 1 with value: 4.3050510008551885.
[I 2025-06-22 14:23:23,246] Trial 2 finished with value: 4.289789115950679 and parameters: {'max_depth': 9, 'learning_rate': 0.020589728197687916, 'subsample': 0.5909124836035503, 'colsample_bytree': 0.591702254926716

Best hyperparameters: {'max_depth': 3, 'learning_rate': 0.05323458539106301, 'subsample': 0.5007447210575431, 'colsample_bytree': 0.569987334762439, 'reg_alpha': 3.972792364171065, 'reg_lambda': 8.996667735732014}
Best MSE: 4.165988793348561


In [22]:

# Prepare the data for XGBoost
dtrain = DMatrix(X_train, label=y_train)
dtest = DMatrix(X_test, label=y_test)

# Use early stopping and the best number of rounds
evals = [(dtrain, "train"), (dtest, "eval")]

best_params = study.best_params

final_model = xgb_train(
    best_params,
    dtrain,
    num_boost_round=10000,
    evals=evals,
    early_stopping_rounds=50,
    verbose_eval=False
)

dtest = DMatrix(X_test)
y_test_pred = final_model.predict(dtest)
print(f"Predictions shape: {y_test_pred.shape}, dtest shape: {X_test.shape}")

mse = mean_squared_error(y_test, y_test_pred)
print(f"Test MSE: {mse:.4f}")


Predictions shape: (2000,), dtest shape: (2000, 5)
Test MSE: 4.1501
